# Inspection

In [1]:
# Import necessary libraries 
from pathlib import Path
import pandas as pd

In [2]:
# -----------------------------
# 1. Display settings
# -----------------------------
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

In [3]:
# -----------------------------
# 2. File paths
# -----------------------------
files = {
    "customers": r"D:\Data_Science\Projects\Portfolio_Projects\E-commerce-Sales-Delivery-and-Customer-Satisfaction\Dataset\Raw\olist_customers_dataset.csv",
    "geolocation": r"D:\Data_Science\Projects\Portfolio_Projects\E-commerce-Sales-Delivery-and-Customer-Satisfaction\Dataset\Raw\olist_geolocation_dataset.csv",
    "order_items": r"D:\Data_Science\Projects\Portfolio_Projects\E-commerce-Sales-Delivery-and-Customer-Satisfaction\Dataset\Raw\olist_order_items_dataset.csv",
    "order_payments": r"D:\Data_Science\Projects\Portfolio_Projects\E-commerce-Sales-Delivery-and-Customer-Satisfaction\Dataset\Raw\olist_order_payments_dataset.csv",
    "order_reviews": r"D:\Data_Science\Projects\Portfolio_Projects\E-commerce-Sales-Delivery-and-Customer-Satisfaction\Dataset\Raw\olist_order_reviews_dataset.csv",
    "orders": r"D:\Data_Science\Projects\Portfolio_Projects\E-commerce-Sales-Delivery-and-Customer-Satisfaction\Dataset\Raw\olist_orders_dataset.csv",
    "products": r"D:\Data_Science\Projects\Portfolio_Projects\E-commerce-Sales-Delivery-and-Customer-Satisfaction\Dataset\Raw\olist_products_dataset.csv",
    "sellers": r"D:\Data_Science\Projects\Portfolio_Projects\E-commerce-Sales-Delivery-and-Customer-Satisfaction\Dataset\Raw\olist_sellers_dataset.csv",
    "category_translation": r"D:\Data_Science\Projects\Portfolio_Projects\E-commerce-Sales-Delivery-and-Customer-Satisfaction\Dataset\Raw\product_category_name_translation.csv"
}

In [4]:
# -----------------------------
# 3. Load all datasets
# -----------------------------
dfs = {}

for name, path in files.items():
    file_path = Path(path)

    if not file_path.exists():
        print(f"[WARNING] File not found: {file_path}")
        continue

    try:
        df = pd.read_csv(file_path)
        dfs[name] = df
        print(f"[LOADED] {name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
    except Exception as e:
        print(f"[ERROR] Could not load {name}: {e}")

print("\nDatasets loaded:", list(dfs.keys()))

[LOADED] customers: 99,441 rows x 5 columns
[LOADED] geolocation: 1,000,163 rows x 5 columns
[LOADED] order_items: 112,650 rows x 7 columns
[LOADED] order_payments: 103,886 rows x 5 columns
[LOADED] order_reviews: 99,224 rows x 7 columns
[LOADED] orders: 99,441 rows x 8 columns
[LOADED] products: 32,951 rows x 9 columns
[LOADED] sellers: 3,095 rows x 4 columns
[LOADED] category_translation: 71 rows x 2 columns

Datasets loaded: ['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'sellers', 'category_translation']


In [5]:
# -----------------------------
# 4. Basic inspection function
# -----------------------------
def inspect_dataframe(name: str, df: pd.DataFrame, sample_rows: int = 3) -> None:
    print("\n" + "=" * 100)
    print(f"DATASET: {name}")
    print("=" * 100)

    # Shape
    print(f"\nShape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

    # Columns
    print("\nColumns:")
    print(df.columns.tolist())

    # Dtypes
    print("\nData types:")
    print(df.dtypes)

    # Sample rows
    print(f"\nFirst {sample_rows} rows:")
    print(df.head(sample_rows))

    # Missing values
    print("\nMissing values:")
    missing = df.isna().sum()
    missing_pct = (df.isna().mean() * 100).round(2)
    missing_summary = pd.DataFrame({
        "missing_count": missing,
        "missing_pct": missing_pct
    }).sort_values(by="missing_count", ascending=False)

    print(missing_summary[missing_summary["missing_count"] > 0] if missing_summary["missing_count"].sum() > 0 else "No missing values.")

    # Duplicate rows
    print(f"\nDuplicate rows: {df.duplicated().sum():,}")

    # Number of unique values
    print("\nUnique values per column:")
    nunique_df = pd.DataFrame({
        "nunique": df.nunique(dropna=False)
    }).sort_values(by="nunique", ascending=False)
    print(nunique_df)

    # Numeric summary
    num_cols = df.select_dtypes(include="number").columns
    if len(num_cols) > 0:
        print("\nNumeric summary:")
        print(df[num_cols].describe().T)

    # Categorical summary
    cat_cols = df.select_dtypes(include=["object", "string", "category"]).columns
    if len(cat_cols) > 0:
        print("\nCategorical columns sample summary:")
        cat_summary = pd.DataFrame({
            "nunique": df[cat_cols].nunique(dropna=False),
            "sample_value_1": [
                df[col].dropna().astype(str).iloc[0] if df[col].dropna().shape[0] > 0 else None
                for col in cat_cols
            ],
            "sample_value_2": [
                df[col].dropna().astype(str).iloc[1] if df[col].dropna().shape[0] > 1 else None
                for col in cat_cols
            ],
        })
        print(cat_summary)    

    # Date-like columns guess    
    date_like_cols = [
        col for col in df.columns
        if any(x in col.lower() for x in ["date", "timestamp", "_at"])
    ]
    print("\nPossible date columns:")
    print(date_like_cols if date_like_cols else "No obvious date columns found.")

    # Memory usage
    mem_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"\nApprox. memory usage: {mem_mb:.2f} MB")


In [6]:
# -----------------------------
# 5. Run inspection for all datasets
# -----------------------------
for name, df in dfs.items():
    inspect_dataframe(name, df)


DATASET: customers

Shape: 99,441 rows x 5 columns

Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Data types:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

First 3 rows:
                        customer_id                customer_unique_id  customer_zip_code_prefix          customer_city customer_state
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0                     14409                 franca             SP
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3                      9790  sao bernardo do campo             SP
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e                      1151              sao paulo             SP

Missing values:
No missing values.

Duplicate rows: 0

Unique values per column:
       

In [7]:
# -----------------------------
# 6. Quick key checks
# -----------------------------
print("\n" + "=" * 100)
print("QUICK ID / KEY-LIKE COLUMN CHECKS")
print("=" * 100)

for name, df in dfs.items():
    key_like_cols = [col for col in df.columns if col.endswith("_id") or col == "id"]
    if key_like_cols:
        print(f"\n{name}:")
        for col in key_like_cols:
            unique_count = df[col].nunique(dropna=False)
            total_count = len(df)
            print(f"  {col}: {unique_count:,} unique values out of {total_count:,} rows")


QUICK ID / KEY-LIKE COLUMN CHECKS

customers:
  customer_id: 99,441 unique values out of 99,441 rows
  customer_unique_id: 96,096 unique values out of 99,441 rows

order_items:
  order_id: 98,666 unique values out of 112,650 rows
  order_item_id: 21 unique values out of 112,650 rows
  product_id: 32,951 unique values out of 112,650 rows
  seller_id: 3,095 unique values out of 112,650 rows

order_payments:
  order_id: 99,440 unique values out of 103,886 rows

order_reviews:
  review_id: 98,410 unique values out of 99,224 rows
  order_id: 98,673 unique values out of 99,224 rows

orders:
  order_id: 99,441 unique values out of 99,441 rows
  customer_id: 99,441 unique values out of 99,441 rows

products:
  product_id: 32,951 unique values out of 32,951 rows

sellers:
  seller_id: 3,095 unique values out of 3,095 rows


In [8]:
# -----------------------------
# 7. Preview potential date parsing
# -----------------------------
print("\n" + "=" * 100)
print("DATE PARSING PREVIEW")
print("=" * 100)

for name, df in dfs.items():    
    date_like_cols = [
        col for col in df.columns
        if any(x in col.lower() for x in ["date", "timestamp", "_at"])
    ]
    if date_like_cols:
        print(f"\n{name}:")
        for col in date_like_cols:
            parsed = pd.to_datetime(df[col], errors="coerce")
            print(
                f"  {col}: non-null={df[col].notna().sum():,}, "
                f"parsed_non_null={parsed.notna().sum():,}, "
                f"min={parsed.min()}, max={parsed.max()}"
            )


DATE PARSING PREVIEW

order_items:
  shipping_limit_date: non-null=112,650, parsed_non_null=112,650, min=2016-09-19 00:15:34, max=2020-04-09 22:35:08

order_reviews:
  review_creation_date: non-null=99,224, parsed_non_null=99,224, min=2016-10-02 00:00:00, max=2018-08-31 00:00:00
  review_answer_timestamp: non-null=99,224, parsed_non_null=99,224, min=2016-10-07 18:32:28, max=2018-10-29 12:27:35

orders:
  order_purchase_timestamp: non-null=99,441, parsed_non_null=99,441, min=2016-09-04 21:15:19, max=2018-10-17 17:30:18
  order_approved_at: non-null=99,281, parsed_non_null=99,281, min=2016-09-15 12:16:38, max=2018-09-03 17:40:06
  order_delivered_carrier_date: non-null=97,658, parsed_non_null=97,658, min=2016-10-08 10:34:01, max=2018-09-11 19:48:28
  order_delivered_customer_date: non-null=96,476, parsed_non_null=96,476, min=2016-10-11 13:46:32, max=2018-10-17 13:22:46
  order_estimated_delivery_date: non-null=99,441, parsed_non_null=99,441, min=2016-09-30 00:00:00, max=2018-11-12 00:00

# Cleaning

In [9]:
# -----------------------------
# 8. Cleaning
# -----------------------------

# Create separate cleaned copies so raw-loaded dfs stay unchanged
cleaned_dfs = {name: df.copy() for name, df in dfs.items()}

# Helper: identify date-like columns
def get_date_like_cols(df: pd.DataFrame) -> list:
    return [
        col for col in df.columns
        if any(x in col.lower() for x in ["date", "timestamp", "_at"])
    ]

# Helper: print basic cleaning summary
def print_cleaning_summary(name: str, before_shape: tuple, after_shape: tuple) -> None:
    print(f"\n{name}:")
    print(f"  before: {before_shape[0]:,} rows x {before_shape[1]:,} columns")
    print(f"  after : {after_shape[0]:,} rows x {after_shape[1]:,} columns")

In [10]:
# -----------------------------
# 8.1 Standardize text columns
# -----------------------------
# Strip leading/trailing spaces from text columns only

for name, df in cleaned_dfs.items():
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in text_cols:
        df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

In [11]:
# -----------------------------
# 8.2 Parse date columns
# -----------------------------
for name, df in cleaned_dfs.items():
    date_like_cols = get_date_like_cols(df)
    for col in date_like_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print("Date parsing completed.")

Date parsing completed.


In [12]:
# -----------------------------
# 8.3 Remove exact duplicate rows
# -----------------------------
for name, df in cleaned_dfs.items():
    before_shape = df.shape
    cleaned_dfs[name] = df.drop_duplicates().copy()
    after_shape = cleaned_dfs[name].shape
    print_cleaning_summary(name, before_shape, after_shape)


customers:
  before: 99,441 rows x 5 columns
  after : 99,441 rows x 5 columns

geolocation:
  before: 1,000,163 rows x 5 columns
  after : 738,332 rows x 5 columns

order_items:
  before: 112,650 rows x 7 columns
  after : 112,650 rows x 7 columns

order_payments:
  before: 103,886 rows x 5 columns
  after : 103,886 rows x 5 columns

order_reviews:
  before: 99,224 rows x 7 columns
  after : 99,224 rows x 7 columns

orders:
  before: 99,441 rows x 8 columns
  after : 99,441 rows x 8 columns

products:
  before: 32,951 rows x 9 columns
  after : 32,951 rows x 9 columns

sellers:
  before: 3,095 rows x 4 columns
  after : 3,095 rows x 4 columns

category_translation:
  before: 71 rows x 2 columns
  after : 71 rows x 2 columns


In [13]:
# -----------------------------
# 8.4 Dataset-specific cleaning
# -----------------------------

# --- geolocation ---
# Keep one row per zip prefix using average lat/lng and first city/state found
# This is much more practical for later joins and avoids the massive duplicates

geo_before = cleaned_dfs["geolocation"].shape

cleaned_dfs["geolocation"] = (
    cleaned_dfs["geolocation"]
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg({
        "geolocation_lat": "mean",
        "geolocation_lng": "mean",
        "geolocation_city": "first",
        "geolocation_state": "first"
    })
)

print_cleaning_summary("geolocation (aggregated by zip prefix)", geo_before, cleaned_dfs["geolocation"].shape)


# --- products ---
# Keep missing category values, but label them for easier analysis later
# Also fill a few numeric nulls only where missing is tiny and safe to keep row usable

products_before = cleaned_dfs["products"].shape

cleaned_dfs["products"]["product_category_name"] = (
    cleaned_dfs["products"]["product_category_name"]
    .fillna("unknown")
)

numeric_product_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in numeric_product_cols:
    if col in cleaned_dfs["products"].columns:
        cleaned_dfs["products"][col] = cleaned_dfs["products"][col].fillna(
            cleaned_dfs["products"][col].median()
        )

print_cleaning_summary("products", products_before, cleaned_dfs["products"].shape)


# --- category translation ---
# No major cleaning needed, but keep as copy for later joins

# Add english category translation into products later if possible
cleaned_dfs["products"] = cleaned_dfs["products"].merge(
    cleaned_dfs["category_translation"],
    on="product_category_name",
    how="left"
)

cleaned_dfs["products"]["product_category_name_english"] = (
    cleaned_dfs["products"]["product_category_name_english"]
    .fillna("unknown")
)


# --- order_reviews ---
# Missing review text is normal; keep it as missing
# Only ensure review_score is integer-friendly if needed

reviews_before = cleaned_dfs["order_reviews"].shape
cleaned_dfs["order_reviews"]["review_score"] = cleaned_dfs["order_reviews"]["review_score"].astype("Int64")
print_cleaning_summary("order_reviews", reviews_before, cleaned_dfs["order_reviews"].shape)


# --- orders ---
# Do not drop missing delivery dates blindly; many are status-related
# Create useful delivery metrics for later analysis
orders_before = cleaned_dfs["orders"].shape

cleaned_dfs["orders"]["delivery_days"] = (
    cleaned_dfs["orders"]["order_delivered_customer_date"] -
    cleaned_dfs["orders"]["order_purchase_timestamp"]
).dt.days

cleaned_dfs["orders"]["estimated_delivery_days"] = (
    cleaned_dfs["orders"]["order_estimated_delivery_date"] -
    cleaned_dfs["orders"]["order_purchase_timestamp"]
).dt.days

cleaned_dfs["orders"]["delivered_on_time"] = (
    cleaned_dfs["orders"]["order_delivered_customer_date"] <=
    cleaned_dfs["orders"]["order_estimated_delivery_date"]
).astype("boolean")

cleaned_dfs["orders"].loc[
    cleaned_dfs["orders"]["order_delivered_customer_date"].isna() |
    cleaned_dfs["orders"]["order_estimated_delivery_date"].isna(),
    "delivered_on_time"
] = pd.NA

print_cleaning_summary("orders", orders_before, cleaned_dfs["orders"].shape)

# --- order_items ---
# Create item total value for later revenue analysis

items_before = cleaned_dfs["order_items"].shape

cleaned_dfs["order_items"]["item_total_value"] = (
    cleaned_dfs["order_items"]["price"] + cleaned_dfs["order_items"]["freight_value"]
)

print_cleaning_summary("order_items", items_before, cleaned_dfs["order_items"].shape)


# --- order_payments ---
# Keep raw payment-level table as-is for payment analysis
# No row removal because one order can have multiple payment rows

payments_before = cleaned_dfs["order_payments"].shape
print_cleaning_summary("order_payments", payments_before, cleaned_dfs["order_payments"].shape)


# --- customers / sellers ---
# No major cleaning needed beyond duplicates/text trimming/date handling already done

print_cleaning_summary("customers", dfs["customers"].shape, cleaned_dfs["customers"].shape)
print_cleaning_summary("sellers", dfs["sellers"].shape, cleaned_dfs["sellers"].shape)
print_cleaning_summary("category_translation", dfs["category_translation"].shape, cleaned_dfs["category_translation"].shape)


geolocation (aggregated by zip prefix):
  before: 738,332 rows x 5 columns
  after : 19,015 rows x 5 columns

products:
  before: 32,951 rows x 9 columns
  after : 32,951 rows x 9 columns

order_reviews:
  before: 99,224 rows x 7 columns
  after : 99,224 rows x 7 columns

orders:
  before: 99,441 rows x 8 columns
  after : 99,441 rows x 11 columns

order_items:
  before: 112,650 rows x 7 columns
  after : 112,650 rows x 8 columns

order_payments:
  before: 103,886 rows x 5 columns
  after : 103,886 rows x 5 columns

customers:
  before: 99,441 rows x 5 columns
  after : 99,441 rows x 5 columns

sellers:
  before: 3,095 rows x 4 columns
  after : 3,095 rows x 4 columns

category_translation:
  before: 71 rows x 2 columns
  after : 71 rows x 2 columns


In [14]:
# -----------------------------
# 8.5 Post-cleaning validation checks
# -----------------------------
print("\n" + "=" * 100)
print("POST-CLEANING VALIDATION")
print("=" * 100)

for name, df in cleaned_dfs.items():
    print(f"\n{name}")
    print(f"shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
    print(f"duplicate rows: {df.duplicated().sum():,}")

    date_like_cols = get_date_like_cols(df)
    if date_like_cols:
        print("date columns:")
        for col in date_like_cols:
            print(f"  - {col}: {df[col].dtype}")

# Key uniqueness checks
print("\n" + "=" * 100)
print("KEY UNIQUENESS CHECKS")
print("=" * 100)

key_checks = {
    "customers": ["customer_id", "customer_unique_id"],
    "orders": ["order_id", "customer_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name", "product_category_name_english"]
}

for name, cols in key_checks.items():
    if name in cleaned_dfs:
        print(f"\n{name}:")
        for col in cols:
            if col in cleaned_dfs[name].columns:
                print(
                    f"  {col}: {cleaned_dfs[name][col].nunique(dropna=False):,} unique values "
                    f"out of {len(cleaned_dfs[name]):,} rows"
                )


POST-CLEANING VALIDATION

customers
shape: 99,441 rows x 5 columns
duplicate rows: 0

geolocation
shape: 19,015 rows x 5 columns
duplicate rows: 0

order_items
shape: 112,650 rows x 8 columns
duplicate rows: 0
date columns:
  - shipping_limit_date: datetime64[us]

order_payments
shape: 103,886 rows x 5 columns
duplicate rows: 0

order_reviews
shape: 99,224 rows x 7 columns
duplicate rows: 0
date columns:
  - review_creation_date: datetime64[us]
  - review_answer_timestamp: datetime64[us]

orders
shape: 99,441 rows x 11 columns
duplicate rows: 0
date columns:
  - order_purchase_timestamp: datetime64[us]
  - order_approved_at: datetime64[us]
  - order_delivered_carrier_date: datetime64[us]
  - order_delivered_customer_date: datetime64[us]
  - order_estimated_delivery_date: datetime64[us]

products
shape: 32,951 rows x 10 columns
duplicate rows: 0

sellers
shape: 3,095 rows x 4 columns
duplicate rows: 0

category_translation
shape: 71 rows x 2 columns
duplicate rows: 0

KEY UNIQUENESS CH

In [15]:
# -----------------------------
# 8.6 Missing values after cleaning
# -----------------------------
print("\n" + "=" * 100)
print("MISSING VALUES AFTER CLEANING")
print("=" * 100)

for name, df in cleaned_dfs.items():
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    print(f"\n{name}:")
    if len(missing) == 0:
        print("  No missing values.")
    else:
        missing_pct = ((missing / len(df)) * 100).round(2)
        missing_summary = pd.DataFrame({
            "missing_count": missing,
            "missing_pct": missing_pct
        })
        print(missing_summary)


MISSING VALUES AFTER CLEANING

customers:
  No missing values.

geolocation:
  No missing values.

order_items:
  No missing values.

order_payments:
  No missing values.

order_reviews:
                        missing_count  missing_pct
review_comment_title            87656        88.34
review_comment_message          58247        58.70

orders:
                               missing_count  missing_pct
order_delivered_customer_date           2965         2.98
delivered_on_time                       2965         2.98
delivery_days                           2965         2.98
order_delivered_carrier_date            1783         1.79
order_approved_at                        160         0.16

products:
  No missing values.

sellers:
  No missing values.

category_translation:
  No missing values.


In [16]:
# -----------------------------
# 8.7 Quick business-ready checks
# -----------------------------
print("\n" + "=" * 100)
print("QUICK BUSINESS CHECKS")
print("=" * 100)

# Orders by status
if "orders" in cleaned_dfs:
    print("\nOrder status distribution:")
    print(cleaned_dfs["orders"]["order_status"].value_counts(dropna=False))

# Payment types
if "order_payments" in cleaned_dfs:
    print("\nPayment type distribution:")
    print(cleaned_dfs["order_payments"]["payment_type"].value_counts(dropna=False))

# Top product categories
if "products" in cleaned_dfs:
    print("\nTop product categories:")
    print(cleaned_dfs["products"]["product_category_name_english"].value_counts(dropna=False).head(10))


QUICK BUSINESS CHECKS

Order status distribution:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Payment type distribution:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

Top product categories:
product_category_name_english
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: count, dtype: int64


In [17]:
cleaned_dfs["orders"]["delivery_delay_days"] = (
    cleaned_dfs["orders"]["order_delivered_customer_date"] -
    cleaned_dfs["orders"]["order_estimated_delivery_date"]
).dt.days

In [18]:
# -----------------------------
# 8.8 Fix integer-like derived columns before export
# -----------------------------
int_like_order_cols = [
    "delivery_days",
    "estimated_delivery_days",
    "delivery_delay_days"
]

for col in int_like_order_cols:
    cleaned_dfs["orders"][col] = cleaned_dfs["orders"][col].astype("Int64")

# Exports

In [19]:
# -----------------------------
# 9. Export cleaned tables
# -----------------------------
processed_dir = Path(
    r"D:\Data_Science\Projects\Portfolio_Projects\E-commerce-Sales-Delivery-and-Customer-Satisfaction\Dataset\Processed"
)

processed_dir.mkdir(parents=True, exist_ok=True)

export_files = {
    "customers": "olist_customers_cleaned.csv",
    "geolocation": "olist_geolocation_cleaned.csv",
    "order_items": "olist_order_items_cleaned.csv",
    "order_payments": "olist_order_payments_cleaned.csv",
    "order_reviews": "olist_order_reviews_cleaned.csv",
    "orders": "olist_orders_cleaned.csv",
    "products": "olist_products_cleaned.csv",
    "sellers": "olist_sellers_cleaned.csv",
    "category_translation": "product_category_name_translation_cleaned.csv"
}

print("\n" + "=" * 100)
print("EXPORT CLEANED TABLES")
print("=" * 100)

for name, filename in export_files.items():
    if name in cleaned_dfs:
        output_path = processed_dir / filename
        cleaned_dfs[name].to_csv(output_path, index=False)
        print(f"[EXPORTED] {name} -> {output_path}")
    else:
        print(f"[SKIPPED] {name} not found in cleaned_dfs")


EXPORT CLEANED TABLES
[EXPORTED] customers -> D:\Data_Science\Projects\Portfolio_Projects\E-commerce-Sales-Delivery-and-Customer-Satisfaction\Dataset\Processed\olist_customers_cleaned.csv
[EXPORTED] geolocation -> D:\Data_Science\Projects\Portfolio_Projects\E-commerce-Sales-Delivery-and-Customer-Satisfaction\Dataset\Processed\olist_geolocation_cleaned.csv
[EXPORTED] order_items -> D:\Data_Science\Projects\Portfolio_Projects\E-commerce-Sales-Delivery-and-Customer-Satisfaction\Dataset\Processed\olist_order_items_cleaned.csv
[EXPORTED] order_payments -> D:\Data_Science\Projects\Portfolio_Projects\E-commerce-Sales-Delivery-and-Customer-Satisfaction\Dataset\Processed\olist_order_payments_cleaned.csv
[EXPORTED] order_reviews -> D:\Data_Science\Projects\Portfolio_Projects\E-commerce-Sales-Delivery-and-Customer-Satisfaction\Dataset\Processed\olist_order_reviews_cleaned.csv
[EXPORTED] orders -> D:\Data_Science\Projects\Portfolio_Projects\E-commerce-Sales-Delivery-and-Customer-Satisfaction\Data